In [1]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree
import sys
sys.path.append("..")
from error_func import error

case_name = "case2"
mesh_name = "coil_box"
solver1 = "moose"
solver2 = "comsol"

sol_moose = np.load(f"../../output/{case_name}/gauss/{case_name}_{solver1}.npy")
sol_comsol = np.genfromtxt(f"../../output/{case_name}/gauss/{case_name}_comsol.txt", delimiter = " ")

print(sol_moose.shape)
print(sol_comsol.shape)

print(sol_moose.shape)
print(sol_comsol.shape)


(69629, 9)
(69629, 11)
(69629, 9)
(69629, 11)


In [2]:
print("Min and max X coord:")
print(np.min(sol_moose[:, 0:3]))
print(np.max(sol_moose[:, 0:3]))

print("Min and max Y coord:")
print(np.min(sol_moose[:, 0:3]))
print(np.max(sol_moose[:, 0:3]))

print("Min and max Z coord:")
print(np.min(sol_moose[:, 0:3]))
print(np.max(sol_moose[:, 0:3]))

print("")

print("Min and max X coord:")
print(np.min(sol_comsol[:, 0:3]))
print(np.max(sol_comsol[:, 0:3]))

print("Min and max Y coord:")
print(np.min(sol_comsol[:, 0:3]))
print(np.max(sol_comsol[:, 0:3]))

print("Min and max Z coord:")
print(np.min(sol_comsol[:, 0:3]))
print(np.max(sol_comsol[:, 0:3]))

Min and max X coord:
-0.099551605255125
0.09795298646876
Min and max Y coord:
-0.099551605255125
0.09795298646876
Min and max Z coord:
-0.099551605255125
0.09795298646876

Min and max X coord:
-0.09955160525513063
0.0979529864687601
Min and max Y coord:
-0.09955160525513063
0.0979529864687601
Min and max Z coord:
-0.09955160525513063
0.0979529864687601


In [3]:
tree = KDTree(sol_moose[:, 0:3])
distances, indices = tree.query(sol_comsol[:, 0:3])

max_dist = np.max(distances)
print(f"Maximum alignment error (distance): {max_dist:.6e}")
if max_dist > 1e-4:
    print("Warning: Large distance detected. Are the geometries identical?")


sol_moose_reordered = sol_moose[indices, :]

np.save(f"../../output/{case_name}/gauss/{case_name}_{solver1}_reordered_to_{solver2}.npy", sol_moose_reordered)

Maximum alignment error (distance): 1.344703e-13


In [4]:
sol_moose = np.load(f"../../output/{case_name}/gauss/{case_name}_{solver1}_reordered_to_{solver2}.npy")
# elec_pot_moose = sol_moose[:, 9]
mag_flux_moose = sol_moose[:, 6:9]
mag_vec_moose = sol_moose[:, 3:6]

sol_comsol = np.genfromtxt(f"../../output/{case_name}/gauss/{case_name}_comsol.txt", delimiter = " ")
# elec_pot_comsol = sol_comsol[:, 9]
mag_flux_comsol = sol_comsol[:, 6:9]
mag_vec_comsol = sol_comsol[:, 3:6]

# print(elec_pot_ngsolve.shape)
# print(elec_pot_comsol.shape)

print(mag_flux_moose.shape)
print(mag_flux_comsol.shape)
print(np.max(mag_flux_comsol))
print(np.max(mag_flux_moose))

print(np.min(mag_flux_comsol))
print(np.min(mag_flux_moose))

print("")

print(mag_vec_moose.shape)
print(mag_vec_comsol.shape)
print(np.max(mag_vec_comsol))
print(np.max(mag_vec_moose))

(69629, 3)
(69629, 3)
95418.05953661224
99859.359259539
-78989.86520297115
-78816.17155469

(69629, 3)
(69629, 3)
1465.5466804021175
131610095865100.0


In [5]:
mesh = sol_comsol[:, 0:3].copy()

In [6]:
print(f"Coordinate errors between {solver1} and {solver2}:")

mesh = error(sol=sol_moose[:, 0:3], sol_ref=sol_comsol[:, 0:3], 
             eps = 1e-12, mesh=mesh, tag="vector", save_tag=None)

Coordinate errors between moose and comsol:

  * Max. absolute error in x direction  : 9.453e-14.
  * Avg. absolute error in x direction  : 2.606e-14.

  * Max. relative error in x direction : 3.968e-06 %.
  * Avg. relative error in x direction : 3.634e-10 %.

  * Max. absolute error in y direction  : 9.714e-14.
  * Avg. absolute error in y direction  : 2.783e-14.

  * Max. relative error in y direction : 1.209e-06 %.
  * Avg. relative error in y direction : 2.947e-10 %.

  * Max. absolute error in z direction  : 9.581e-14.
  * Avg. absolute error in z direction  : 2.571e-14.

  * Max. relative error in z direction : 4.411e-06 %.
  * Avg. relative error in z direction : 8.938e-10 %.


In [7]:
print(f"Magnetic flux density errors between {solver1} and {solver2}:")

mesh = error(sol=mag_flux_moose, sol_ref=mag_flux_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag=None, save_tag_array=True)

# print(mag_flux_moose[1100, :])
# print(mag_flux_comsol[1100, :])


lim_x = (mesh[:, 3] > 10.0).sum()
lim_y = (mesh[:, 4] > 10.0).sum()
lim_z = (mesh[:, 5] > 10.0).sum()

print("")

print(f"Percent of values above 10% in x direction: {np.round(lim_x * 100 / mesh.shape[0], 2)}%")
print(f"Percent of values above 10% in y direction: {np.round(lim_y * 100 / mesh.shape[0], 2)}%")
print(f"Percent of values above 10% in z direction: {np.round(lim_z * 100 / mesh.shape[0], 2)}%")

B_x_err = mesh[:, 3]
B_y_err = mesh[:, 4]
B_z_err = mesh[:, 5]

B_x_err = np.extract(B_x_err<=10, B_x_err)
B_y_err = np.extract(B_y_err<=10, B_y_err)
B_z_err = np.extract(B_z_err<=10, B_z_err)



Magnetic flux density errors between moose and comsol:

  * Max. absolute error in x direction  : 4.007e+04.
  * Avg. absolute error in x direction  : 8.765e+02.

  * Max. relative error in x direction : 1.278e+05 %.
  * Avg. relative error in x direction : 4.246e+01 %.

  * Max. absolute error in y direction  : 7.371e+03.
  * Avg. absolute error in y direction  : 3.410e+02.

  * Max. relative error in y direction : 8.488e+05 %.
  * Avg. relative error in y direction : 7.966e+01 %.

  * Max. absolute error in z direction  : 4.402e+04.
  * Avg. absolute error in z direction  : 5.013e+03.

  * Max. relative error in z direction : 1.771e+06 %.
  * Avg. relative error in z direction : 2.311e+02 %.

Percent of values above 10% in x direction: 37.34%
Percent of values above 10% in y direction: 34.66%
Percent of values above 10% in z direction: 84.96%


/home/wiera/Documents/EM_simulation/scripts/case2/../error_func.py:62: RuntimeWarning: divide by zero encountered in divide
  rel_error_dir = np.where(denom > eps, num * 100 / denom, np.nan)
/home/wiera/Documents/EM_simulation/scripts/case2/../error_func.py:62: RuntimeWarning: invalid value encountered in divide
  rel_error_dir = np.where(denom > eps, num * 100 / denom, np.nan)


In [8]:
print(f"Magnetic vector potential errors between {solver1} and {solver2}:")

mesh = error(sol=mag_vec_moose, sol_ref=mag_vec_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag=None)

Magnetic vector potential errors between moose and comsol:

  * Max. absolute error in x direction  : 1.177e+14.
  * Avg. absolute error in x direction  : 7.905e+11.

  * Max. relative error in x direction : 1.734e+16 %.
  * Avg. relative error in x direction : 8.812e+12 %.

  * Max. absolute error in y direction  : 1.172e+14.
  * Avg. absolute error in y direction  : 1.332e+12.

  * Max. relative error in y direction : 4.232e+14 %.
  * Avg. relative error in y direction : 3.773e+11 %.

  * Max. absolute error in z direction  : 1.316e+14.
  * Avg. absolute error in z direction  : 6.076e+11.

  * Max. relative error in z direction : 3.834e+16 %.
  * Avg. relative error in z direction : 9.041e+12 %.


In [9]:
np.save(f"../../output/{case_name}/gauss/{case_name}_error_{solver1}_{solver2}.npy", mesh)
np.savetxt(f"../../output/{case_name}/gauss/{case_name}_error_{solver1}_{solver2}.txt", mesh, delimiter=',', header='x,y,z,B_x_err,B_y_err,B_z_err')